# AutoGen Memory & RAG

## Course Notebook for Students

**Goal:** Learn how AutoGen agents remember useful context and answer questions using retrieved knowledge.

This notebook is designed for teaching. It includes theory, runnable examples, real-world mini projects, and exercises.

### What students will build

1. A simple customer-preference memory.
2. An AutoGen assistant using `ListMemory`.
3. A from-scratch RAG pipeline students can understand line by line.
4. A store FAQ RAG assistant for real-world business queries.
5. An AutoGen RAG-style assistant using memory.
6. A ChromaDB vector memory example.
7. A small evaluation workflow for retrieved chunks and grounded answers.

> Note: AutoGen is currently in maintenance mode, but its Memory and RAG patterns are still valuable for learning agent design.

## 0. Prerequisites

Students should know:

- Python basics
- Lists, dictionaries, functions
- Basic LLM concepts
- What embeddings and vector search mean at a high level
- Jupyter Notebook basics

### Recommended setup

- Python 3.10+
- Jupyter Notebook or VS Code notebook
- OpenAI API key stored as `OPENAI_API_KEY` for LLM cells

Many cells run without an API key. LLM cells are clearly marked.

In [3]:
# Run this cell once in a fresh environment.
# If you already installed these packages, you can skip it.

%pip install -U "autogen-agentchat" "autogen-ext[openai]" pandas scikit-learn chromadb sentence-transformers typing_extensions

INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 3.1 MB/s eta 0:00:03
   ------ --------------------------------- 1.3/8.2 MB 2.8 MB/s eta 0:00:03
   ---------- ----------------------------- 2.1/8.2 MB 2.9 MB/s eta 0:00:03
   ------------ --------------------------- 2.6/8.2 MB 2.8 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 3.0 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.2 MB 3.4 MB/s eta 0:00:02
   -------------------------- ------------- 5.5/8.2 MB 3.6 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.2 MB 3.7 MB/s eta 0:00:01
   ------------------------------------- -- 7.6/8.2 MB 3.9 MB/s eta 0:00:01
   ---------------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 0.2.0 requires chromadb!=0.5.10,!=0.5.11,!=0.5.12,!=0.5.4,!=0.5.5,!=0.5.7,!=0.5.9,<0.6.0,>=0.4.0, but you have chromadb 1.5.9 which is incompatible.
langchain-chroma 0.2.0 requires langchain-core!=0.3.0,!=0.3.1,!=0.3.10,!=0.3.11,!=0.3.12,!=0.3.13,!=0.3.14,!=0.3.2,!=0.3.3,!=0.3.4,!=0.3.5,!=0.3.6,!=0.3.7,!=0.3.8,!=0.3.9,<0.4.0,>=0.2.43, but you have langchain-core 1.2.7 which is incompatible.
opentelemetry-instrumentation 0.50b0 requires opentelemetry-semantic-conventions==0.50b0, but you have opentelemetry-semantic-conventions 0.64b0 which is incompatible.
opentelemetry-instrumentation-asgi 0.50b0 requires opentelemetry-semantic-conventions==0.50b0, but you have opentelemetry-semantic-conventions 0.64b0 which is incompatible.
opentelemetry-instrumentation-fastap

In [4]:
import os
import sys
from pathlib import Path

print("Python:", sys.version)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

WORK_DIR = Path("autogen_memory_rag_workspace")
WORK_DIR.mkdir(exist_ok=True)
print("Workspace:", WORK_DIR.resolve())

Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
OPENAI_API_KEY set: False
Workspace: C:\Users\LotusBlue\Coding\GenAI\Course\autogen_memory_rag_workspace


## 1. Memory vs RAG: Simple Mental Model

### Memory

Memory stores useful facts that should influence future responses.

Examples:

- User prefers Kannada replies.
- Customer likes cotton kurtis.
- Store offers alteration for selected products.
- The project uses Python 3.12.

### RAG

RAG means **Retrieval-Augmented Generation**.

Instead of asking the model to answer from memory alone, we:

1. Load documents.
2. Split them into chunks.
3. Store chunks in a searchable index.
4. Retrieve relevant chunks for a question.
5. Ask the LLM to answer using retrieved context.

### Difference

Memory is usually about useful facts and history. RAG is usually about retrieving from documents, manuals, policies, PDFs, databases, or knowledge bases.

## 2. AutoGen Memory Concepts

AutoGen AgentChat provides a `Memory` protocol with these important operations:

- `add`: add new entries
- `query`: retrieve relevant entries
- `update_context`: add retrieved information to the agent context
- `clear`: clear memory entries
- `close`: clean up resources

For teaching, `ListMemory` is the easiest starting point. It stores memories in a simple list and appends them to the agent context.

In [7]:
from autogen_core.memory import ListMemory, MemoryContent, MemoryMimeType


user_memory = ListMemory()

await user_memory.add(
    MemoryContent(
        content="The customer prefers cotton and breathable fabrics.",
        mime_type=MemoryMimeType.TEXT,
        metadata={"type": "preference", "domain": "retail"},
    )
)

await user_memory.add(
    MemoryContent(
        content="The customer prefers replies in simple English with Kannada terms only when useful.",
        mime_type=MemoryMimeType.TEXT,
        metadata={"type": "communication_preference"},
    )
)

await user_memory.add(
    MemoryContent(
        content="The store's current offer is Buy 3 Get 1 on selected kurti and salwar sets.",
        mime_type=MemoryMimeType.TEXT,
        metadata={"type": "offer"},
    )
)

query_result = await user_memory.query("What should I remember about this customer?")
print(query_result)

results=[MemoryContent(content='The customer prefers cotton and breathable fabrics.', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata={'type': 'preference', 'domain': 'retail'}), MemoryContent(content='The customer prefers replies in simple English with Kannada terms only when useful.', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata={'type': 'communication_preference'}), MemoryContent(content="The store's current offer is Buy 3 Get 1 on selected kurti and salwar sets.", mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata={'type': 'offer'})]


## 3. AutoGen Assistant with ListMemory

This example shows how memory changes an agent's response.

The user does not repeat their preference in the task. The agent should still use the memory.

In [9]:
import os
from autogen_ext.models.openai import OpenAIChatCompletionClient


def require_openai_key() -> None:
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError(
            "OPENAI_API_KEY is not set. Set it before running LLM cells."
        )


def create_model_client(model: str = "gpt-4o-mini") -> OpenAIChatCompletionClient:
    require_openai_key()
    return OpenAIChatCompletionClient(model=model)

In [10]:
# LLM cell: requires OPENAI_API_KEY.

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console


memory_agent = AssistantAgent(
    name="memory_retail_agent",
    model_client=create_model_client(),
    memory=[user_memory],
    system_message=(
        "You are a retail shopping assistant. Use memory when it is relevant. "
        "Give practical, concise suggestions."
    ),
)

await Console(
    memory_agent.run_stream(
        task="Recommend 3 outfit types for a customer visiting the shop in summer."
    )
)

OSError: OPENAI_API_KEY is not set. Set it before running LLM cells.

### Classroom discussion

Ask students:

1. Which memory was useful?
2. Which memory was irrelevant?
3. What happens if memory contains wrong information?
4. Should every conversation be saved as memory?

Good memory systems are selective. They remember stable, useful facts, not every small message.

## 4. Build a Simple RAG Pipeline From Scratch

Before using vector databases, students should understand the pipeline.

This local example uses TF-IDF retrieval. It is not as powerful as embeddings, but it is transparent, fast, and easy to teach.

Use case:

> A boutique wants a chatbot that answers from store policy and product FAQ documents.

In [ ]:
store_docs = [
    {
        "source": "exchange_policy.txt",
        "text": "Exchange is allowed within 7 days with original bill and product tag. Used, washed, altered, or damaged items are not eligible for exchange.",
    },
    {
        "source": "alteration_policy.txt",
        "text": "Basic alteration is available for selected kurtis and salwar sets. Alteration usually takes 2 to 4 working days depending on tailor availability.",
    },
    {
        "source": "offers.txt",
        "text": "Current offer is Buy 3 Get 1 on selected kurti and salwar sets. The free item will be the lowest priced eligible item.",
    },
    {
        "source": "fabric_guide.txt",
        "text": "Cotton kurtis are breathable and suitable for summer. Rayon has a soft drape. Linen blends look premium but may wrinkle more easily.",
    },
    {
        "source": "care_instructions.txt",
        "text": "Hand wash or gentle machine wash is recommended for embroidered kurtis. Avoid harsh bleach and dry garments in shade.",
    },
    {
        "source": "store_timing.txt",
        "text": "Store timing is 10:30 AM to 9:00 PM from Monday to Saturday and 11:00 AM to 8:00 PM on Sunday.",
    },
]

print("Documents:", len(store_docs))
for doc in store_docs:
    print("-", doc["source"])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd


class SimpleTfidfRetriever:
    def __init__(self, documents: list[dict]):
        self.documents = documents
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.matrix = self.vectorizer.fit_transform([doc["text"] for doc in documents])

    def search(self, query: str, k: int = 3) -> list[dict]:
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix).flatten()
        ranked_indices = scores.argsort()[::-1][:k]

        results = []
        for idx in ranked_indices:
            results.append(
                {
                    "source": self.documents[idx]["source"],
                    "text": self.documents[idx]["text"],
                    "score": round(float(scores[idx]), 4),
                }
            )
        return results


retriever = SimpleTfidfRetriever(store_docs)
results = retriever.search("Can I exchange a kurti after purchase?", k=3)
pd.DataFrame(results)

In [ ]:
def build_grounded_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"Source: {chunk['source']}\nContent: {chunk['text']}"
        for chunk in retrieved_chunks
    )
    return f'''
You are a store FAQ assistant.
Answer the question using only the context below.
If the context does not contain the answer, say: "I do not have that information in the store documents."

Context:
{context}

Question:
{question}

Answer:
'''.strip()


question = "If I buy 3 kurtis, which item is free?"
retrieved = retriever.search(question, k=2)
prompt = build_grounded_prompt(question, retrieved)

print(prompt)

In [ ]:
# LLM cell: requires OPENAI_API_KEY.
# This manually performs RAG: retrieve first, then send context to the model.

from autogen_core.models import UserMessage

model_client = create_model_client()
response = await model_client.create([UserMessage(content=prompt, source="user")])
print(response.content)
await model_client.close()

## 5. RAG as an AutoGen Tool

Another practical pattern is to expose retrieval as a tool.

The agent can call `search_store_knowledge` whenever it needs store policy or FAQ context.

In [ ]:
def search_store_knowledge(query: str) -> str:
    """Search store policy and FAQ documents for relevant context."""
    matches = retriever.search(query, k=3)
    return "\n\n".join(
        f"Source: {match['source']} | Score: {match['score']}\n{match['text']}"
        for match in matches
    )


print(search_store_knowledge("What is the alteration time?"))

In [ ]:
# LLM cell: requires OPENAI_API_KEY.

rag_tool_agent = AssistantAgent(
    name="store_rag_agent",
    model_client=create_model_client(),
    tools=[search_store_knowledge],
    system_message=(
        "You are a store FAQ assistant. "
        "Use the search_store_knowledge tool before answering policy, offer, fabric, timing, or care questions. "
        "Ground your answer in retrieved context and mention when information is not available."
    ),
    reflect_on_tool_use=True,
)

await Console(
    rag_tool_agent.run_stream(
        task="A customer asks: Can I exchange an altered kurti, and how long does alteration take?"
    )
)

## 6. AutoGen Memory as RAG Context

AutoGen memory can retrieve useful information and add it to an agent's model context.

For small teaching examples, `ListMemory` works well. For larger document collections, use vector memory such as ChromaDB.

In [ ]:
store_memory = ListMemory()

for doc in store_docs:
    await store_memory.add(
        MemoryContent(
            content=doc["text"],
            mime_type=MemoryMimeType.TEXT,
            metadata={"source": doc["source"], "kind": "store_document"},
        )
    )

memory_query = await store_memory.query("What is the current Buy 3 Get 1 offer?")
print(memory_query)

In [ ]:
# LLM cell: requires OPENAI_API_KEY.

memory_rag_agent = AssistantAgent(
    name="memory_rag_agent",
    model_client=create_model_client(),
    memory=[store_memory],
    system_message=(
        "You are a store assistant. Use memory as your source of truth. "
        "If memory does not contain the answer, say you do not have that information."
    ),
)

await Console(
    memory_rag_agent.run_stream(
        task="Explain the current offer and exchange rule in simple words."
    )
)

## 7. ChromaDB Vector Memory

For larger RAG systems, we need semantic search. `ChromaDBVectorMemory` stores text chunks in a vector database and retrieves semantically similar chunks.

This section may download a sentence-transformer model the first time it runs.

If your environment is slow or offline, skip this section and use the TF-IDF retriever above.

In [ ]:
# Optional vector memory example.
# First run may download embedding model files.

RUN_CHROMA_EXAMPLE = False

if RUN_CHROMA_EXAMPLE:
    from autogen_ext.memory.chromadb import (
        ChromaDBVectorMemory,
        PersistentChromaDBVectorMemoryConfig,
        SentenceTransformerEmbeddingFunctionConfig,
    )

    chroma_memory = ChromaDBVectorMemory(
        config=PersistentChromaDBVectorMemoryConfig(
            collection_name="store_policy_memory",
            persistence_path=str(WORK_DIR / "chromadb_store_policy"),
            k=3,
            score_threshold=0.2,
            embedding_function_config=SentenceTransformerEmbeddingFunctionConfig(
                model_name="all-MiniLM-L6-v2"
            ),
        )
    )

    await chroma_memory.clear()

    for doc in store_docs:
        await chroma_memory.add(
            MemoryContent(
                content=doc["text"],
                mime_type=MemoryMimeType.TEXT,
                metadata={"source": doc["source"]},
            )
        )

    vector_results = await chroma_memory.query("Can a washed kurti be exchanged?")
    print(vector_results)

    await chroma_memory.close()
else:
    print("ChromaDB example skipped. Set RUN_CHROMA_EXAMPLE = True to run it.")

In [ ]:
# LLM cell + optional ChromaDB.
# Requires OPENAI_API_KEY and RUN_CHROMA_EXAMPLE = True in the previous cell.

RUN_CHROMA_AGENT_EXAMPLE = False

if RUN_CHROMA_AGENT_EXAMPLE:
    from autogen_ext.memory.chromadb import (
        ChromaDBVectorMemory,
        PersistentChromaDBVectorMemoryConfig,
        SentenceTransformerEmbeddingFunctionConfig,
    )

    chroma_memory = ChromaDBVectorMemory(
        config=PersistentChromaDBVectorMemoryConfig(
            collection_name="store_policy_memory_agent",
            persistence_path=str(WORK_DIR / "chromadb_store_policy_agent"),
            k=3,
            score_threshold=0.2,
            embedding_function_config=SentenceTransformerEmbeddingFunctionConfig(
                model_name="all-MiniLM-L6-v2"
            ),
        )
    )
    await chroma_memory.clear()
    for doc in store_docs:
        await chroma_memory.add(
            MemoryContent(
                content=doc["text"],
                mime_type=MemoryMimeType.TEXT,
                metadata={"source": doc["source"]},
            )
        )

    chroma_agent = AssistantAgent(
        name="chroma_rag_agent",
        model_client=create_model_client(),
        memory=[chroma_memory],
        system_message=(
            "Answer store questions only from retrieved memory. "
            "If context is missing, say you do not have enough information."
        ),
    )

    await Console(
        chroma_agent.run_stream(task="Can I exchange a washed kurti?")
    )
    await chroma_memory.close()
else:
    print("Chroma agent example skipped.")

## 8. Chunking Strategy

RAG quality depends heavily on chunking.

Bad chunking causes:

- Missing context
- Mixed topics in one chunk
- Poor retrieval
- Hallucinated answers

### Common chunking strategies

- Fixed-size chunks
- Paragraph-based chunks
- Heading-based chunks
- Sliding window chunks with overlap
- Semantic chunks

For business documents, heading-based or paragraph-based chunking often works well.

In [ ]:
long_policy_text = '''
# Exchange Policy
Exchange is allowed within 7 days with original bill and product tag.
Used, washed, altered, or damaged items are not eligible.

# Alteration Policy
Basic alteration is available for selected products.
Alteration usually takes 2 to 4 working days.

# Fabric Care
Hand wash embroidered products.
Dry garments in shade.
Avoid harsh bleach.
'''


def split_by_heading(text: str) -> list[dict]:
    chunks = []
    current_heading = "General"
    current_lines = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("# "):
            if current_lines:
                chunks.append(
                    {"heading": current_heading, "text": " ".join(current_lines)}
                )
            current_heading = line.replace("# ", "").strip()
            current_lines = []
        else:
            current_lines.append(line)

    if current_lines:
        chunks.append({"heading": current_heading, "text": " ".join(current_lines)})

    return chunks


heading_chunks = split_by_heading(long_policy_text)
pd.DataFrame(heading_chunks)

## 9. RAG Evaluation Basics

RAG systems should be evaluated on two separate things:

1. **Retrieval quality**: Did we fetch the right chunks?
2. **Answer quality**: Did the final answer stay faithful to the retrieved chunks?

Simple classroom metrics:

- Top-k retrieval hit
- Source coverage
- Answer contains citation/source
- No-answer behavior when context is missing

In [ ]:
test_cases = [
    {
        "question": "Can I exchange a washed item?",
        "expected_source": "exchange_policy.txt",
    },
    {
        "question": "How many days does alteration take?",
        "expected_source": "alteration_policy.txt",
    },
    {
        "question": "Which fabric is good for summer?",
        "expected_source": "fabric_guide.txt",
    },
    {
        "question": "What is the free item in Buy 3 Get 1?",
        "expected_source": "offers.txt",
    },
]


def evaluate_retriever(retriever: SimpleTfidfRetriever, cases: list[dict], k: int = 3) -> pd.DataFrame:
    rows = []
    for case in cases:
        results = retriever.search(case["question"], k=k)
        retrieved_sources = [r["source"] for r in results]
        rows.append(
            {
                "question": case["question"],
                "expected_source": case["expected_source"],
                "retrieved_sources": retrieved_sources,
                "hit": case["expected_source"] in retrieved_sources,
            }
        )
    return pd.DataFrame(rows)


evaluation_df = evaluate_retriever(retriever, test_cases, k=3)
evaluation_df

In [ ]:
accuracy = evaluation_df["hit"].mean()
print(f"Top-3 retrieval hit rate: {accuracy:.0%}")

## 10. Guardrails for Memory and RAG

### Memory guardrails

- Do not store sensitive personal data unless necessary.
- Do not remember temporary facts as permanent preferences.
- Let users inspect or delete memory in production systems.
- Add metadata: source, date, category, confidence.

### RAG guardrails

- Answer only from retrieved context for policy/legal/medical/financial topics.
- Say "I do not know" when context is missing.
- Show source names where possible.
- Log retrieved chunks for debugging.
- Evaluate retrieval separately from generation.

### Common failure modes

- Stale documents
- Poor chunking
- Wrong top-k
- Irrelevant chunks
- Model ignores context
- Context has conflicting information

## 11. Mini Project 1: Boutique FAQ Bot

Build an AutoGen assistant that answers questions from store documents.

### Requirements

1. Add at least 8 documents.
2. Implement retriever.
3. Expose retriever as a tool.
4. Ask 5 customer questions.
5. Include source names in answers.

Example questions:

- Can I exchange an altered kurti?
- What fabric is best for summer?
- What is the current offer?
- How should I wash embroidered kurtis?
- What time is the shop open on Sunday?

## 12. Mini Project 2: Personal Shopping Memory

Create a memory-based shopping assistant.

### Memory examples

- Customer prefers cotton.
- Customer avoids sleeveless.
- Customer budget is below ₹2,000.
- Customer likes pastel colors.
- Customer wants office wear.

### Task

Ask the agent:

> Recommend 3 outfits for this customer.

The answer should use memory without the user repeating preferences.

## 13. Mini Project 3: Enterprise RAG

Build a RAG assistant for one of these domains:

- HR policy assistant
- QA automation documentation assistant
- Cloud security runbook assistant
- School admission FAQ assistant
- Product catalogue assistant

### Expected design

1. Documents
2. Chunking
3. Indexing
4. Retrieval
5. Grounded answer generation
6. Evaluation test cases

## 14. Capstone Assignment

Build a **Memory + RAG Agent**.

### Requirements

1. Use memory for user preferences.
2. Use RAG for document knowledge.
3. Use at least one AutoGen `AssistantAgent`.
4. Include one retrieval tool or vector memory.
5. Include at least 5 evaluation questions.
6. Show what happens when the answer is not in the documents.

### Deliverables

- Working notebook
- Explanation of memory design
- Explanation of RAG pipeline
- Retrieval evaluation table
- Screenshots or outputs from 3 successful queries

## 15. Reference Links

- AutoGen GitHub: https://github.com/microsoft/autogen
- AutoGen Memory and RAG: https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/memory.html
- AutoGen AgentChat agents: https://microsoft.github.io/autogen/stable/reference/python/autogen_agentchat.agents.html
- AutoGen ChromaDB memory API: https://microsoft.github.io/autogen/dev/reference/python/autogen_ext.memory.chromadb.html
- ChromaDB: https://www.trychroma.com/